# Day 07 · 文件 I/O 与异常

> **前置**: Day01-06 已掌握变量、字符串、分支、循环、函数、列表字典
> **关键问题**: 让程序**持久化数据到磁盘**,并学会用异常处理让程序**不因报错就崩溃**

**引入(5 分钟)**

**第 1 讲:文件读写基础 —— with open / read / readline / write**

Python 推荐**`with` 语句**,出块自动 `close()`,即便中间报错也不会漏关。三种基础模式:`'r'` 读(默认)、`'w'` 写(覆盖)、`'a'` 追加。口诀:**写 `'w'` 会覆盖,追加用 `'a'`,读完记得 `close()`**。三种读法:`read()` 整个文件 → 一个字符串;`readline()` 只读一行;`readlines()` 全部读入 → 每行作为字符串的列表。

In [ ]:
# ===== 写文件(覆盖模式 'w') =====
# with 语句:出块后文件自动关闭,无需手动 close()
with open("diary.txt", "w", encoding="utf-8") as f:
    f.write("2026-07-08 晴\n")      # write 写入一行,\n 是换行符
    f.write("今天学了文件 I/O\n")
    f.write("with 语句真方便\n")
# 出 with 块,f 已自动关闭,数据已刷到磁盘

# ===== 读文件(默认 'r') =====
with open("diary.txt", "r", encoding="utf-8") as f:
    content = f.read()               # read():一次性读完全部,返回一个字符串
print(content)                      # 打印全部内容

# ===== 追加模式 'a':不清空原内容,在末尾追加 =====
with open("diary.txt", "a", encoding="utf-8") as f:
    f.write("追加一行\n")

# ===== 读法 2:readlines() 返回每行作为字符串的列表 =====
with open("diary.txt", "r", encoding="utf-8") as f:
    rows = f.readlines()             # 每行作为字符串的列表
print("行数:", len(rows))
print("第一行:", rows[0].strip())   # strip() 去掉末尾换行符

---

**第 2 讲:JSON 读写 —— json.load / json.dump**

纯文本 `"苹果,香蕉,橘子"` 无法表达**结构化数据**(列表/字典嵌套)。**JSON** 是程序之间交换数据的"通用语言",Python 内置 `json` 模块。口诀:**`dump` 写入文件,`load` 读出文件;末尾的 `s` = string(字符串)**。

In [ ]:
import json

# ===== 准备数据:Python 字典包含列表嵌套 =====
data = {
    "name": "小明",
    "scores": [90, 85, 92],
    "is_vip": True
}

# ===== json.dump:把 Python 对象写入 JSON 文件 =====
# ensure_ascii=False:中文不转成 unicode 转义形式,保持可读
# indent=2:美化缩进,方便人看
with open("data.json", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

# ===== json.load:从 JSON 文件读回 Python 对象 =====
with open("data.json", "r", encoding="utf-8") as f:
    loaded = json.load(f)            # loaded 是一个字典,与 data 结构相同
print(loaded["scores"])             # [90, 85, 92]
print(type(loaded))                 # <class 'dict'>

# ===== json.dumps / json.loads:字符串 ←→ Python 对象 =====
# dumps:把 Python 对象转成 JSON 字符串
s = json.dumps(data, ensure_ascii=False)
print(type(s))                      # <class 'str'>

# loads:把 JSON 字符串解析成 Python 对象
obj = json.loads(s)
print(obj["name"])                  # 小明

---

**第 3 讲:异常处理 —— try / except / else / finally**

程序遇到错误会**崩溃退出**。`try-except` 让我们"抓住"错误,决定下一步,而不是直接死掉。口诀:**`try` 里放危险代码,`except` 按类型分流,`else` 无错才跑,`finally` 必定执行**(常用来关文件/清理资源)。

常见异常:`FileNotFoundError`(文件不存在)、`ValueError`(`int("abc")` 转换失败)、`KeyError`(访问字典不存在的 key)、`IndexError`(列表越界)。**永远指定具体异常类型,别裸 `except:`,那会吞掉真正的 Bug**。

In [ ]:
# ===== try / except / else / finally 完整语法 =====
# 下面演示正常执行的场景

try:
    num = int("42")               # 正常执行,不触发异常
    result = 100 / num
except ValueError:
    print("输入的不是整数!")        # 输入不是整数时走这里
except ZeroDivisionError:
    print("不能除以 0!")            # 除数为 0 时走这里
except Exception as e:
    print(f"未知错误: {e}")         # 兜底:捕获其他所有异常(放在最后)
else:
    print("计算结果是:", result)    # 没有异常时才执行 else 块
finally:
    print("无论有无异常都执行")     # 无论有无异常都执行(常用于关文件、释放资源)

print("程序继续往下跑,没有崩溃!")  # 异常被捕获,程序不会中断

In [ ]:
# ===== 实战:用异常处理读 JSON 文件,崩不了 =====
import json, os

FILE = "diary.json"

def safe_read_json(path):
    """安全读取 JSON 文件,文件不存在或内容非法都返回空列表"""
    if not os.path.exists(path):
        return []                    # 文件不存在 -> 返回空列表
    with open(path, "r", encoding="utf-8") as f:
        try:
            return json.load(f)      # 尝试解析 JSON
        except json.JSONDecodeError:
            return []                # 内容非法 -> 返回空列表

# 测试 1:文件不存在
print("不存在的文件:", safe_read_json("not_exist.json"))   # []

# 测试 2:创建合法 JSON 后读取
data = ["今天天气不错", "学了文件读写"]
with open(FILE, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False)
print("正常读取:", safe_read_json(FILE))

# 测试 3:文件内容非法
with open(FILE, "w", encoding="utf-8") as f:
    f.write("这不是 JSON")       # 写入非法内容
print("非法内容:", safe_read_json(FILE))                   # []

---

**今日小结**

三大块:`with open` 安全读写文件、`json.load/dump` 持久化结构化数据、`try-except-else-finally` 异常防护。

易错点:漏写 `encoding="utf-8"`,中文路径直接报 `UnicodeDecodeError`;`'w'` 模式会覆盖原文件,慎用;裸 `except:` 吞掉所有异常包括 `Ctrl+C`,必须指定具体异常类型。

**更多练习**

- 当堂练:course/lesson07/in_class/practice01-06.py
- 课后作业:course/lesson07/homework/task01-03.py